# Scaling laws: hands-on (optional)
**LLMs From Basics · Lesson M2.4**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · Fit a power law in log-log space
Below is synthetic data: the final loss of 12 training runs at different compute budgets $C$ (in FLOPs), with a little noise. We suspect a power law $L = a\,C^{-b}$.

Taking $\log_{10}$ of both sides gives a straight line: $\log_{10} L = \log_{10} a - b\,\log_{10} C$. So fit a line (`np.polyfit(x, y, 1)` returns `[slope, intercept]`) to the logged data and convert back.

In [ ]:
def true_loss(C):                     # hidden "truth" (don't peek in your fit!)
    return 30.0 * C ** -0.05
C_data = np.logspace(17, 21, 12)
L_data = true_loss(C_data) * np.exp(rng.normal(0, 0.005, C_data.size))
plt.loglog(C_data, L_data, "o"); plt.xlabel("compute C (FLOPs)"); plt.ylabel("loss"); plt.title("Loss vs compute (log-log)"); plt.show()

In [ ]:
def fit_power_law(C, L):
    # TODO: fit log10(L) = log10(a) - b * log10(C); return (a, b) with b > 0
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
a, b = fit_power_law(C_data, L_data)
print(f"fitted: L = {a:.2f} · C^-{b:.4f}")
assert abs(b - 0.05) < 0.003 and 20 < a < 45
a2, b2 = fit_power_law(np.array([1e2, 1e4]), np.array([1.0, 0.1]))
assert np.isclose(b2, 0.5) and np.isclose(a2, 10.0)
print("✓ exercise 1")

## 2 · Extrapolate
The whole point of a scaling law: predict a run **1,000× bigger** than anything you trained. Use your fit to predict the loss at $C = 10^{24}$ FLOPs.

In [ ]:
def predict(C, a, b):
    # TODO: the fitted power law
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
pred = predict(1e24, a, b)
print(f"predicted loss at 1e24 FLOPs: {pred:.3f}   (truth: {true_loss(1e24):.3f})")
assert abs(pred / true_loss(1e24) - 1) < 0.03
Cs = np.logspace(17, 24, 50)
plt.loglog(C_data, L_data, "o", label="runs"); plt.loglog(Cs, predict(Cs, a, b), "--", label="fit")
plt.axvline(1e24, color="gray", lw=0.8); plt.legend(); plt.xlabel("C"); plt.ylabel("loss"); plt.show()
print("✓ exercise 2")

## 3 · Counting FLOPs: $C \approx 6ND$
About $2N$ FLOPs per token forward and $4N$ backward, for $N$ parameters and $D$ tokens. Also convert FLOPs into GPU-hours: an H100 peaks near $989\times10^{12}$ dense BF16 FLOP/s, and real runs reach some fraction of that, the **MFU** (model FLOPs utilization).

In [ ]:
def train_flops(N, D):
    # TODO
    raise NotImplementedError("Exercise 3a")

def gpu_hours(C, peak=989e12, mfu=0.4):
    # TODO: seconds on one GPU at peak*mfu FLOP/s, converted to hours
    raise NotImplementedError("Exercise 3b")

**Check 3**

In [ ]:
assert np.isclose(train_flops(7e9, 140e9), 5.88e21)
assert np.isclose(train_flops(405e9, 15.6e12), 3.79e25, rtol=1e-3)   # Llama 3 405B: Meta reports ~3.8e25
assert 4000 < gpu_hours(5.88e21) < 4200
print(f"7B on 140B tokens: {gpu_hours(5.88e21):,.0f} H100-hours at 40% MFU")
print(f"Llama 3 405B: {gpu_hours(train_flops(405e9, 15.6e12)) / 1e6:.1f} million H100-hours at 40% MFU")
print("✓ exercise 3")

## 4 · A Chinchilla calculator
With $D = k N$ tokens ($k \approx 20$) and $C = 6ND = 6kN^2$, the compute-optimal size is $N = \sqrt{C / 6k}$.

In [ ]:
def chinchilla(C, tokens_per_param=20):
    # TODO: return (N_opt, D_opt)
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
N, D = chinchilla(5.88e21)
assert np.isclose(N, 7e9) and np.isclose(D, 1.4e11)
N, D = chinchilla(1.2e24)
assert np.isclose(N, 1e11) and np.isclose(D, 2e12)
N1, _ = chinchilla(1e22); N2, _ = chinchilla(1e23)
assert np.isclose(N2 / N1, np.sqrt(10)), "10x compute should mean ~3.16x parameters"
for name, n, d in [("GPT-3", 175e9, 300e9), ("Chinchilla", 70e9, 1.4e12), ("Llama 2 7B", 7e9, 2e12), ("Llama 3 8B", 8e9, 15e12)]:
    print(f"{name:11s} {d / n:7.1f} tokens/param   compute-optimal size for its budget: {chinchilla(train_flops(n, d))[0] / 1e9:6.1f}B")
print("✓ exercise 4")

## 5 · IsoFLOP curves: find the best size numerically
Use the Chinchilla-style form $L(N, D) = E + A/N^{\alpha} + B/D^{\beta}$. These constants are **illustrative** (chosen so the optimum is exactly 20 tokens per parameter); real fitted values differ. For a fixed budget $C$, set $D = C/6N$ and scan $N$.

In [ ]:
E, A, ALPHA = 1.69, 406.4, 0.34
B, BETA = A * 20 ** ALPHA, ALPHA
def L_ND(N, D): return E + A / N ** ALPHA + B / D ** BETA

In [ ]:
def best_size(C, Ns=np.logspace(7, 12, 2001)):
    # TODO: compute the loss for every N in Ns with D = C / (6N); return (best N, its loss)
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
for C in [1e19, 1e21, 1e23]:
    n_num, _ = best_size(C)
    n_rule, _ = chinchilla(C)
    print(f"C = {C:.0e}: numerical best N = {n_num / 1e9:7.2f}B   20-tokens rule = {n_rule / 1e9:7.2f}B")
    assert abs(np.log10(n_num / n_rule)) < 0.01
Ns = np.logspace(7.5, 11.5, 300)
for C in [1e19, 1e20, 1e21, 1e22]:
    plt.semilogx(Ns, L_ND(Ns, C / (6 * Ns)), label=f"C = {C:.0e}")
    n, l = best_size(C); plt.plot(n, l, "o", color="k")
plt.ylim(1.9, 3.5); plt.xlabel("parameters N"); plt.ylabel("loss"); plt.legend(); plt.title("IsoFLOP curves"); plt.show()
print("✓ exercise 5")

## Experiment
- Make the data in exercise 1 noisier (`0.005` → `0.03`) or fit only the 4 smallest runs. How far off is the extrapolation to $10^{24}$ now? This is why labs train a *ladder* of runs across many orders of magnitude.
- In exercise 5, how much worse (in %) is the loss at $C = 10^{22}$ if you train a model **4× smaller** than optimal on 4× more tokens? That's the price of "overtraining" for cheaper inference.
- Change `ALPHA` and `BETA` to different values (e.g. 0.34 and 0.28). Does the best tokens-per-parameter ratio stay constant as $C$ grows?

In [ ]:
# Your experiments here
C = 1e22
n_opt, l_opt = best_size(C)
l_small = L_ND(n_opt / 4, C / (6 * n_opt / 4))
print(f"4x smaller model: {100 * (l_small / l_opt - 1):.2f}% higher loss, but ~4x cheaper per generated token")

## Recap
- A power law is a straight line on log-log axes: fit it with linear regression on the logs, then extrapolate.
- Training compute $C \approx 6ND$; divide by achieved FLOP/s for GPU-time.
- Compute-optimal (Chinchilla): $N, D \propto \sqrt{C}$, about 20 tokens per parameter.
- Near the optimum the IsoFLOP curve is flat, so a smaller, longer-trained model costs only a little loss.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def fit_power_law(C, L):
    slope, intercept = np.polyfit(np.log10(C), np.log10(L), 1)
    return 10 ** intercept, -slope

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def predict(C, a, b):
    return a * C ** -b

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def train_flops(N, D):
    return 6 * N * D

def gpu_hours(C, peak=989e12, mfu=0.4):
    return C / (peak * mfu) / 3600

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def chinchilla(C, tokens_per_param=20):
    N = np.sqrt(C / (6 * tokens_per_param))
    return N, tokens_per_param * N

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def best_size(C, Ns=np.logspace(7, 12, 2001)):
    losses = L_ND(Ns, C / (6 * Ns))
    i = np.argmin(losses)
    return Ns[i], losses[i]